# Training Script!

This is the jupyter notebook of finetuning script of my Qwen-2.5-VL Notebook.
Run all the cells one by one.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MAX_JOBS"] = "4"   # bounds the flash-attn build so it doesn't stall/exhaust RAM

!pip install -q transformers==4.49.0 accelerate peft hqq "qwen-vl-utils[decord]" scikit-learn pandas matplotlib seaborn huggingface_hub tqdm
!pip install -q flash-attn --no-build-isolation --no-cache-dir

In [2]:
from huggingface_hub import notebook_login
notebook_login()   # paste your HF write-token when the widget appears

In [3]:
CONFIG = {
    "model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
    "hf_dataset_repo": "may-ur08/Cricket_Dataset",
    "dataset_root": "/teamspace/studios/this_studio/hf_download",
    "manifest_dir": "/teamspace/studios/this_studio/hf_download/manifests",
    "hf_repo_id": "may-ur08/qwen2.5-vl-gen-cricket-commentary",
    "fps": 2.0,
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.1,
    "lr": 5e-5, "weight_decay": 0.01,
    "num_epochs": 15,              # ← back up from 7 — early stopping (patience) actually enforces the real ceiling now, no need to guess low
    "batch_size": 4,               # ← up from 1 — A10G's 24GB forced batch_size=1; H100's 80GB comfortably fits 4
    "grad_accum_steps": 4,         # ← down from 16 — keeps effective batch size at 16, same as before, just reached differently
    "grad_clip": 1.0,
    "warmup_pct": 0.05, "num_workers": 0, "seed": 42,   # num_workers stays 0 — see note below, this wasn't a memory issue
    "push_interval": 2,
    "output_dir": "/teamspace/studios/this_studio/checkpoints",
    "min_acceptable_f1": 0.35,
    "small_class_threshold": 160, "small_class_val": 15, "small_class_test": 15,
    "max_frames": 16,              # ← up from 8 — more temporal detail per clip, was capped purely for A10G memory
    "max_pixels": 401408,          # ← up from 200704 — roughly ~633x633/frame, clearer visual detail per frame
    "patience": 3,
}

In [4]:
from huggingface_hub import snapshot_download
import zipfile, os

download_path = snapshot_download(
    repo_id=CONFIG["hf_dataset_repo"],
    repo_type="dataset",
    local_dir="/teamspace/studios/this_studio/hf_download",
)
print("Downloaded to:", download_path)
print("Contents:", os.listdir(download_path))

# If you uploaded a zip, extract it; if you uploaded the folder structure directly, skip extraction
zip_candidates = [f for f in os.listdir(download_path) if f.endswith(".zip")]
if zip_candidates:
    zip_path = os.path.join(download_path, zip_candidates[0])
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(CONFIG["dataset_root"])
    print(f"Extracted {zip_candidates[0]} -> {CONFIG['dataset_root']}")
else:
    # Folder structure was uploaded directly — just point dataset_root at the download
    CONFIG["dataset_root"] = download_path
    print("No zip found — using downloaded folder structure directly as dataset_root")

Fetching 2727 files:   0%|          | 0/2727 [00:00<?, ?it/s]

outcome/bowled/(10).mp4:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

outcome/bowled/(100).mp4:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

manifest.csv: 0.00B [00:00, ?B/s]

...

shot/cover/cover_0162.mp4:   0%|          | 0.00/476k [00:00<?, ?B/s]

shot/cover/cover_0164.mp4:   0%|          | 0.00/742k [00:00<?, ?B/s]

shot/cover/cover_0163.mp4:   0%|          | 0.00/559k [00:00<?, ?B/s]

shot/cover/cover_0177.mp4:   0%|          | 0.00/940k [00:00<?, ?B/s]

HfHubHTTPError: 429 Client Error: Too Many Requests for url: https://huggingface.co/api/datasets/may-ur08/Cricket_Dataset/xet-read-token/a78c47f8daf7ab23eff30773b87214e289483101 (Request ID: Root=1-6a527e84-147009ce4e71877073c9cce3;051b2fa5-6407-48d1-9c80-240b348b3bdf)

We had to rate limit you, you hit the quota of 1000 api requests per 5 minutes period. Upgrade to a PRO user or Team/Enterprise organization account (https://hf.co/pricing) to get higher limits. See https://huggingface.co/docs/hub/rate-limits

In [5]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # ← avoids the endpoint that's been hitting 429s

import time
from huggingface_hub import list_repo_files, hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

local_dir = "/teamspace/studios/this_studio/hf_download"
os.makedirs(local_dir, exist_ok=True)

print("Fetching file list from repo...")
all_files = list_repo_files(repo_id=CONFIG["hf_dataset_repo"], repo_type="dataset")
print(f"Total files in repo: {len(all_files)}")

# Only keep files that don't already exist locally (or exist but are suspiciously tiny — likely a failed/partial download)
MIN_VALID_SIZE = 1024  # bytes — anything smaller than 1KB almost certainly isn't a real video file
to_download = []
for f in all_files:
    local_path = os.path.join(local_dir, f)
    if not os.path.exists(local_path) or os.path.getsize(local_path) < MIN_VALID_SIZE:
        to_download.append(f)

print(f"Already present and valid: {len(all_files) - len(to_download)}")
print(f"Remaining to download: {len(to_download)}")

failed = []
for i, filename in enumerate(to_download):
    retries = 0
    while retries < 5:
        try:
            hf_hub_download(
                repo_id=CONFIG["hf_dataset_repo"],
                repo_type="dataset",
                filename=filename,
                local_dir=local_dir,
            )
            break
        except HfHubHTTPError as e:
            if "429" in str(e):
                wait = 30 * (retries + 1)   # backs off: 30s, 60s, 90s, 120s, 150s
                print(f"Rate limited on {filename} — waiting {wait}s (retry {retries+1}/5)")
                time.sleep(wait)
                retries += 1
            else:
                print(f"Failed on {filename}: {e}")
                failed.append(filename)
                break
    else:
        failed.append(filename)

    if i % 50 == 0:
        print(f"Progress: {i}/{len(to_download)}")

print(f"\nDone. Failed files: {len(failed)}")
if failed:
    print(failed[:10])

Fetching file list from repo...
Total files in repo: 2727
Already present and valid: 1004
Remaining to download: 1723


shot/cover/cover_0165.mp4:   0%|          | 0.00/846k [00:00<?, ?B/s]

Progress: 0/1723


shot/cover/cover_0166.mp4:   0%|          | 0.00/658k [00:00<?, ?B/s]

shot/cover/cover_0167.mp4:   0%|          | 0.00/724k [00:00<?, ?B/s]

...

shot/sweep/sweep_0191.mp4:   0%|          | 0.00/778k [00:00<?, ?B/s]

shot/sweep/sweep_0192.mp4:   0%|          | 0.00/729k [00:00<?, ?B/s]

shot/sweep/sweep_0193.mp4:   0%|          | 0.00/777k [00:00<?, ?B/s]

shot/sweep/sweep_0194.mp4:   0%|          | 0.00/1.02M [00:00<?, ?B/s]


Done. Failed files: 4
['shot/late_cut/late_cut_0104.mp4', 'shot/late_cut/late_cut_0110.mp4', 'shot/lofted/lofted_0081.mp4', 'shot/lofted/lofted_0082.mp4']


In [6]:
import os
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import HfHubHTTPError
import time

local_dir = "/teamspace/studios/this_studio/hf_download"
still_failed = ['shot/late_cut/late_cut_0104.mp4', 'shot/late_cut/late_cut_0110.mp4',
                 'shot/lofted/lofted_0081.mp4', 'shot/lofted/lofted_0082.mp4']

for filename in still_failed:
    try:
        hf_hub_download(
            repo_id=CONFIG["hf_dataset_repo"],
            repo_type="dataset",
            filename=filename,
            local_dir=local_dir,
        )
        print(f"✅ Downloaded: {filename}")
    except HfHubHTTPError as e:
        print(f"❌ Still failing: {filename} — {e}")
    time.sleep(5)   # small gap between requests, avoids re-triggering the rate limit

shot/late_cut/late_cut_0104.mp4:   0%|          | 0.00/452k [00:00<?, ?B/s]

✅ Downloaded: shot/late_cut/late_cut_0104.mp4


shot/late_cut/late_cut_0110.mp4:   0%|          | 0.00/1.13M [00:00<?, ?B/s]

✅ Downloaded: shot/late_cut/late_cut_0110.mp4


shot/lofted/lofted_0081.mp4:   0%|          | 0.00/884k [00:00<?, ?B/s]

✅ Downloaded: shot/lofted/lofted_0081.mp4


shot/lofted/lofted_0082.mp4:   0%|          | 0.00/918k [00:00<?, ?B/s]

✅ Downloaded: shot/lofted/lofted_0082.mp4


In [7]:
import json
with open(f"{CONFIG['manifest_dir']}/train.json") as f:
    sample = json.load(f)
print(sample[0])

{'id': 'shot_Reverse Sweep_00001', 'video': '/content/drive/MyDrive/cricket_dataset/shot/Reverse Sweep/(10).mp4', 'conversations': [{'from': 'human', 'value': '<video>\nWhat cricket shot is being played in this video?'}, {'from': 'gpt', 'value': 'Reverse Sweep'}], 'task': 'shot', 'label': 'Reverse Sweep', 'weight': 0.10425720702853739}


In [8]:
import json, pandas as pd

OLD_PREFIX = "/content/drive/MyDrive/cricket_dataset"
NEW_PREFIX = CONFIG["dataset_root"]   # "/teamspace/studios/this_studio/hf_download"

def fix_manifest_json(path):
    with open(path) as f:
        data = json.load(f)

    for item in data:
        item["video"] = item["video"].replace(OLD_PREFIX, NEW_PREFIX)
        item["label"] = item["label"].lower().strip()
        item["conversations"][1]["value"] = item["conversations"][1]["value"].lower().strip()

    with open(path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Fixed {path} — {len(data)} entries")
    print("Sample after fix:", data[0])

for split in ["train", "val", "test"]:
    fix_manifest_json(f"{CONFIG['manifest_dir']}/{split}.json")

# Same fix for manifest.csv, if you want it kept consistent too
csv_path = f"{CONFIG['manifest_dir']}/manifest.csv"
df = pd.read_csv(csv_path)
df["filepath"] = df["filepath"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
df["label"] = df["label"].str.lower().str.strip()
df.to_csv(csv_path, index=False)
print(f"\nFixed {csv_path}")
print(df.head(3))

Fixed /teamspace/studios/this_studio/hf_download/manifests/train.json — 2180 entries
Sample after fix: {'id': 'shot_Reverse Sweep_00001', 'video': '/teamspace/studios/this_studio/hf_download/shot/Reverse Sweep/(10).mp4', 'conversations': [{'from': 'human', 'value': '<video>\nWhat cricket shot is being played in this video?'}, {'from': 'gpt', 'value': 'reverse sweep'}], 'task': 'shot', 'label': 'reverse sweep', 'weight': 0.10425720702853739}
Fixed /teamspace/studios/this_studio/hf_download/manifests/val.json — 271 entries
Sample after fix: {'id': 'shot_Reverse Sweep_00004', 'video': '/teamspace/studios/this_studio/hf_download/shot/Reverse Sweep/(105).mp4', 'conversations': [{'from': 'human', 'value': '<video>\nWhat cricket shot is being played in this video?'}, {'from': 'gpt', 'value': 'reverse sweep'}], 'task': 'shot', 'label': 'reverse sweep', 'weight': 1.0}
Fixed /teamspace/studios/this_studio/hf_download/manifests/test.json — 271 entries
Sample after fix: {'id': 'shot_Reverse Sweep_

In [9]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, HqqConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quant_config = HqqConfig(nbits=4, group_size=64)

try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("Using flash_attention_2")
except ImportError:
    attn_impl = "sdpa"
    print("flash-attn not available, falling back to sdpa")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CONFIG["model_id"],
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
)
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM",
))

# Gradient checkpointing trades speed for memory — with 80GB available and a moderate batch size,
# you likely don't need this trade anymore. Leaving it enabled is still safe (just slightly slower);
# disabling it is optional if you want to test whether it meaningfully speeds up epochs.
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.print_trainable_parameters()

Using flash_attention_2


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


trainable params: 10,092,544 || all params: 4,702,350,336 || trainable%: 0.2146


In [10]:
import json, random
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class CricketDataset(Dataset):
    def __init__(self, json_path, processor, fps):
        with open(json_path) as f:
            self.data = json.load(f)
        self.processor = processor
        self.fps = fps

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        max_retries = 5
        for attempt in range(max_retries):
            item = self.data[idx]
            try:
                question = item["conversations"][0]["value"].replace("<video>\n", "")
                answer = item["conversations"][1]["value"]

                messages = [{"role": "user", "content": [
                    {"type": "video", "video": item["video"], "fps": self.fps},
                    {"type": "text", "text": question},
                ]}]
                prompt_text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                full_text = prompt_text + answer + self.processor.tokenizer.eos_token
                _, video_inputs = process_vision_info(messages)

                prompt_ids = self.processor(text=[prompt_text], videos=video_inputs, return_tensors="pt")["input_ids"][0]
                full_enc = self.processor(text=[full_text], videos=video_inputs, return_tensors="pt")

                labels = full_enc["input_ids"][0].clone()
                labels[:len(prompt_ids)] = -100

                return {
                    "input_ids": full_enc["input_ids"][0], "attention_mask": full_enc["attention_mask"][0],
                    "pixel_values_videos": full_enc.get("pixel_values_videos"), "video_grid_thw": full_enc.get("video_grid_thw"),
                    "labels": labels, "task": item["task"], "label": item["label"], "weight": item["weight"],
                }
            except Exception as e:
                print(f"⚠️  Skipping unreadable video: {item['video']} ({type(e).__name__}: {e})")
                idx = random.randint(0, len(self.data) - 1)   # try a different random sample instead

        raise RuntimeError(f"Failed to load a valid sample after {max_retries} retries")

In [15]:
import decord

bad_files = []
for item in train_ds.data + val_ds.data:
    try:
        decord.VideoReader(item["video"])
    except Exception as e:
        bad_files.append((item["video"], str(e)))

print(f"Found {len(bad_files)} unreadable files out of {len(train_ds.data) + len(val_ds.data)}")
for path, err in bad_files[:20]:
    print(path)

Found 0 unreadable files out of 2451


In [12]:
import os
print(os.path.exists(f"{CONFIG['manifest_dir']}/train.json"))

True


In [13]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import WeightedRandomSampler, DataLoader

def collate_fn(batch):
    return {
        "input_ids": pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=processor.tokenizer.pad_token_id),
        "attention_mask": pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0),
        "labels": pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100),
        "pixel_values_videos": torch.cat([b["pixel_values_videos"] for b in batch], dim=0),
        "video_grid_thw": torch.cat([b["video_grid_thw"] for b in batch], dim=0),
        "meta": [{"task": b["task"], "label": b["label"]} for b in batch],
    }

train_ds = CricketDataset(f"{CONFIG['manifest_dir']}/train.json", processor, CONFIG["fps"])
val_ds   = CricketDataset(f"{CONFIG['manifest_dir']}/val.json", processor, CONFIG["fps"])

sampler = WeightedRandomSampler([it["weight"] for it in train_ds.data], num_samples=len(train_ds.data), replacement=True)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], sampler=sampler, collate_fn=collate_fn, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)

print(f"train: {len(train_ds)} examples | val: {len(val_ds)} examples")

train: 2180 examples | val: 271 examples


In [14]:
sample = train_ds[0]
print(sample["input_ids"].shape, sample["pixel_values_videos"].shape if sample["pixel_values_videos"] is not None else None)

qwen-vl-utils using decord to read video.


torch.Size([1054]) torch.Size([4080, 1176])


In [16]:
import torch
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report, matthews_corrcoef, f1_score)

@torch.no_grad()
def evaluate(loader, compute_loss=False):
    model.eval()
    preds, golds = {"shot": [], "outcome": []}, {"shot": [], "outcome": []}
    val_loss_total, val_loss_count = 0.0, 0

    for batch in loader:
        meta = batch["meta"][0]
        if compute_loss:
            batch_gpu = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items() if k != "meta"}
            val_loss_total += model(**batch_gpu).loss.item()
            val_loss_count += 1

        gen = model.generate(
            input_ids=batch["input_ids"].to(model.device), attention_mask=batch["attention_mask"].to(model.device),
            pixel_values_videos=batch["pixel_values_videos"].to(model.device), video_grid_thw=batch["video_grid_thw"].to(model.device),
            max_new_tokens=12)
        pred_text = processor.batch_decode(gen[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        pred_text = pred_text.strip().split("\n")[0].split(":")[0].strip().lower()
        preds[meta["task"]].append(pred_text)
        golds[meta["task"]].append(meta["label"])

    results = {}
    for task in ["shot", "outcome"]:
        if not golds[task]: continue
        valid_labels = sorted(set(golds[task]))
        p, r, f1, _ = precision_recall_fscore_support(golds[task], preds[task], labels=valid_labels, average="macro", zero_division=0)
        weighted_f1 = f1_score(golds[task], preds[task], labels=valid_labels, average="weighted", zero_division=0)
        bal_acc = balanced_accuracy_score(golds[task], preds[task]) if set(preds[task]) & set(valid_labels) else 0.0
        mcc = matthews_corrcoef(golds[task], preds[task]) if len(set(preds[task])) > 1 else 0.0
        per_class_f1 = dict(zip(valid_labels, f1_score(golds[task], preds[task], labels=valid_labels, average=None, zero_division=0)))
        results[task] = {
            "accuracy": accuracy_score(golds[task], preds[task]), "balanced_accuracy": bal_acc,
            "macro_precision": p, "macro_recall": r, "macro_f1": f1, "weighted_f1": weighted_f1, "mcc": mcc,
            "per_class_f1": per_class_f1,
            "confusion_matrix": confusion_matrix(golds[task], preds[task], labels=valid_labels).tolist(),
            "labels_order": valid_labels,
            "classification_report": classification_report(golds[task], preds[task], labels=valid_labels, zero_division=0, output_dict=True),
        }
    model.train()
    val_loss = val_loss_total / val_loss_count if val_loss_count else None
    return results, val_loss

In [17]:
import os, json, torch
from huggingface_hub import upload_file
from transformers import get_cosine_schedule_with_warmup
from tqdm.notebook import tqdm

os.makedirs(CONFIG["output_dir"], exist_ok=True)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
total_steps = (len(train_loader) // CONFIG["grad_accum_steps"]) * CONFIG["num_epochs"]
scheduler = get_cosine_schedule_with_warmup(optimizer, int(CONFIG["warmup_pct"] * total_steps), total_steps)

history = []
best_val_score, best_epoch_dir, patience_counter = -1, None, 0

for epoch in range(1, CONFIG["num_epochs"] + 1):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}")):
        batch_gpu = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items() if k != "meta"}
        loss = model(**batch_gpu).loss / CONFIG["grad_accum_steps"]
        loss.backward()
        running_loss += loss.item()

        if (step + 1) % CONFIG["grad_accum_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), CONFIG["grad_clip"])
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            torch.cuda.empty_cache()

        if step % 20 == 0:
            print(f"  step {step}/{len(train_loader)} | running loss so far: {running_loss/(step+1):.4f}")

    val_metrics, val_loss = evaluate(val_loader, compute_loss=True)
    min_val_f1 = min(v["macro_f1"] for v in val_metrics.values())
    avg_mcc = sum(v["mcc"] for v in val_metrics.values()) / len(val_metrics)

    epoch_log = {
        "epoch": epoch, "train_loss": running_loss / len(train_loader), "val_loss": val_loss,
        "lr": scheduler.get_last_lr()[0], "min_val_f1": min_val_f1, "avg_mcc": avg_mcc,
        "val_metrics": val_metrics,
    }
    history.append(epoch_log)
    print(f"Epoch {epoch}: train_loss={epoch_log['train_loss']:.4f} | val_loss={val_loss:.4f} | "
          f"shot_f1={val_metrics['shot']['macro_f1']:.3f} | outcome_f1={val_metrics['outcome']['macro_f1']:.3f} | "
          f"min_f1={min_val_f1:.3f} | avg_mcc={avg_mcc:.3f}")

    for task, m in val_metrics.items():
        if m["macro_f1"] < 0.2:
            print(f"⚠️  Task '{task}' macro-F1 very low ({m['macro_f1']:.3f}) at epoch {epoch}")

    if min_val_f1 > best_val_score:
        best_val_score = min_val_f1
        patience_counter = 0
        best_epoch_dir = f"{CONFIG['output_dir']}/best_epoch{epoch}"
        model.save_pretrained(best_epoch_dir); processor.save_pretrained(best_epoch_dir)
        with open(f"{best_epoch_dir}/metrics.json", "w") as f:
            json.dump(epoch_log, f, indent=2)
        flag = "⚠️ below acceptable quality" if best_val_score < CONFIG["min_acceptable_f1"] else "✅"
        print(f"{flag} New best saved at epoch {epoch}: min_val_f1={best_val_score:.3f}")
    else:
        patience_counter += 1
        print(f"No improvement — patience {patience_counter}/{CONFIG['patience']}")
        if patience_counter >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    if epoch % CONFIG["push_interval"] == 0 and best_epoch_dir:
        model.push_to_hub(CONFIG["hf_repo_id"], commit_message=f"checkpoint epoch {epoch}, min_val_f1={best_val_score:.3f}")
        processor.push_to_hub(CONFIG["hf_repo_id"])
        with open("/tmp/training_history.json", "w") as f:
            json.dump(history, f, indent=2)
        upload_file(path_or_fileobj="/tmp/training_history.json", path_in_repo="training_history.json", repo_id=CONFIG["hf_repo_id"])

Epoch 1:   0%|          | 0/545 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


  step 0/545 | running loss so far: 2.9991


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


  step 20/545 | running loss so far: 2.8673
  step 40/545 | running loss so far: 2.8510
  step 60/545 | running loss so far: 2.8837
  step 80/545 | running loss so far: 2.9133
  step 100/545 | running loss so far: 2.9014
  step 120/545 | running loss so far: 2.8705
  step 140/545 | running loss so far: 2.8177
  step 160/545 | running loss so far: 2.7328
  step 180/545 | running loss so far: 2.5852
  step 200/545 | running loss so far: 2.4275
  step 220/545 | running loss so far: 2.2715
  step 240/545 | running loss so far: 2.1293
  step 260/545 | running loss so far: 2.0054
  step 280/545 | running loss so far: 1.8920
  step 300/545 | running loss so far: 1.7919
  step 320/545 | running loss so far: 1.6986


OutOfMemoryError: CUDA out of memory. Tried to allocate 22.91 GiB. GPU 0 has a total capacity of 79.18 GiB of which 17.12 GiB is free. Process 9343 has 62.05 GiB memory in use. Of the allocated memory 61.17 GiB is allocated by PyTorch, and 151.16 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from peft import PeftModel

best_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(CONFIG["model_id"], quantization_config=quant_config, device_map="auto", torch_dtype=torch.bfloat16)
best_model = PeftModel.from_pretrained(best_model, best_epoch_dir)
test_ds = CricketDataset(f"{CONFIG['manifest_dir']}/test.json", processor, CONFIG["fps"])
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
model = best_model
test_metrics = evaluate(test_loader)

quality_note = ""
if test_metrics["shot"]["macro_f1"] < CONFIG["min_acceptable_f1"] or test_metrics["outcome"]["macro_f1"] < CONFIG["min_acceptable_f1"]:
    quality_note = "\n⚠️ **Note**: One or both tasks are below the acceptable quality threshold. Treat as a baseline, not final.\n"

card = f"""---
base_model: {CONFIG['model_id']}
tags: [cricket, video-classification, qwen2.5-vl, lora, hqq]
---
# Qwen2.5-VL Cricket Shot & Outcome Classifier

Fine-tuned with HQQ 4-bit quantization + LoRA (r={CONFIG['lora_r']}, attention-only, ViT frozen).

## Test Set Results
- Shot: accuracy={test_metrics['shot']['accuracy']:.3f}, macro-F1={test_metrics['shot']['macro_f1']:.3f}, MCC={test_metrics['shot']['mcc']:.3f}
- Outcome: accuracy={test_metrics['outcome']['accuracy']:.3f}, macro-F1={test_metrics['outcome']['macro_f1']:.3f}, MCC={test_metrics['outcome']['mcc']:.3f}
{quality_note}

## Training Curves
![training curves](charts/training_curves.png)

## Confusion Matrices (Test Set)
![shot confusion matrix](charts/confmat_shot_test.png)
![outcome confusion matrix](charts/confmat_outcome_test.png)

## Per-Class F1 (Test Set)
![shot per-class f1](charts/per_class_f1_shot_test.png)
![outcome per-class f1](charts/per_class_f1_outcome_test.png)

## Dataset Class Distribution
![class distribution](charts/class_distribution.png)
"""

with open("/tmp/README.md", "w") as f: f.write(card)
with open("/tmp/test_metrics.json", "w") as f: json.dump(test_metrics, f, indent=2)
best_model.push_to_hub(CONFIG["hf_repo_id"], commit_message="final model")
processor.push_to_hub(CONFIG["hf_repo_id"])
upload_file(path_or_fileobj="/tmp/README.md", path_in_repo="README.md", repo_id=CONFIG["hf_repo_id"])
upload_file(path_or_fileobj="/tmp/test_metrics.json", path_in_repo="test_metrics.json", repo_id=CONFIG["hf_repo_id"])
print(json.dumps(test_metrics, indent=2))

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(epochs, [h["train_loss"] for h in history], label="Train Loss", marker="o")
axes[0,0].plot(epochs, [h["val_loss"] for h in history], label="Val Loss", marker="o")
axes[0,0].set_title("Training vs Validation Loss"); axes[0,0].set_xlabel("Epoch"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(epochs, [h["val_metrics"]["shot"]["macro_f1"] for h in history], label="Shot macro-F1", marker="o")
axes[0,1].plot(epochs, [h["val_metrics"]["outcome"]["macro_f1"] for h in history], label="Outcome macro-F1", marker="o")
axes[0,1].plot(epochs, [h["min_val_f1"] for h in history], label="min(F1) — selection criterion", marker="s", linestyle="--", color="black")
axes[0,1].set_title("Validation F1 by Task"); axes[0,1].set_xlabel("Epoch"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(epochs, [h["avg_mcc"] for h in history], label="Avg MCC", marker="o", color="green")
axes[1,0].plot(epochs, [h["min_val_f1"] for h in history], label="min(F1)", marker="o", color="orange")
axes[1,0].set_title("MCC vs F1 — do they agree?"); axes[1,0].set_xlabel("Epoch"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(epochs, [h["lr"] for h in history], marker="o", color="purple")
axes[1,1].set_title("Learning Rate Schedule"); axes[1,1].set_xlabel("Epoch"); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_curves.png", dpi=150)
plt.show()

In [ ]:
import seaborn as sns

def plot_confusion_heatmap(metrics_dict, task, split_name):
    cm = metrics_dict[task]["confusion_matrix"]
    labels = metrics_dict[task]["labels_order"]
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"{task.capitalize()} Confusion Matrix — {split_name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/confmat_{task}_{split_name}.png", dpi=150)
    plt.show()

# Usage after running evaluate() on test_loader:
# test_metrics, _ = evaluate(test_loader)
# plot_confusion_heatmap(test_metrics, "shot", "test")
# plot_confusion_heatmap(test_metrics, "outcome", "test")

In [ ]:
def plot_per_class_f1(metrics_dict, task, split_name):
    per_class = metrics_dict[task]["per_class_f1"]
    labels, scores = list(per_class.keys()), list(per_class.values())
    plt.figure(figsize=(10, 5))
    bars = plt.bar(labels, scores, color="steelblue")
    plt.axhline(metrics_dict[task]["macro_f1"], color="red", linestyle="--", label=f"Macro-F1 avg ({metrics_dict[task]['macro_f1']:.3f})")
    plt.title(f"{task.capitalize()} Per-Class F1 — {split_name}")
    plt.xticks(rotation=45, ha="right"); plt.ylabel("F1 Score"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/per_class_f1_{task}_{split_name}.png", dpi=150)
    plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv(f"{CONFIG['manifest_dir']}/manifest.csv")
dist = df.groupby(["task", "label", "split"]).size().unstack(fill_value=0)
dist.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="viridis")
plt.title("Class Distribution Across Train / Val / Test Splits")
plt.ylabel("Number of Clips"); plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/class_distribution.png", dpi=150)
plt.show()

In [ ]:
import os
from huggingface_hub import upload_file

def push_all_artifacts_to_hf(output_dir, repo_id):
    chart_files = [f for f in os.listdir(output_dir) if f.endswith(".png")]
    if not chart_files:
        print("⚠️ No .png files found in output_dir — did you run the chart cells first?")
        return

    for fname in chart_files:
        local_path = os.path.join(output_dir, fname)
        upload_file(
            path_or_fileobj=local_path,
            path_in_repo=f"charts/{fname}",
            repo_id=repo_id,
            commit_message=f"add chart: {fname}",
        )
        print(f"✅ Uploaded charts/{fname}")

push_all_artifacts_to_hf(CONFIG["output_dir"], CONFIG["hf_repo_id"])